In [ ]:
"""
Medical Image Classification - PyTorch Implementation
Robust conversion from TensorFlow/Keras + sklearn pipeline
Features: MobileNetV3, data splitting, augmentation, feature extraction,
          and multiple ML classifiers (KNN, LR, SVM, DT, RF) + CNN transfer learning
"""

import os
import gc
import time
import shutil
import pickle
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from PIL import Image

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import torchvision.transforms.functional as TF

# Scikit-learn imports
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler, label_binarize
from sklearn.model_selection import (
    train_test_split, StratifiedShuffleSplit, GridSearchCV,
    RandomizedSearchCV, cross_val_score
)
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_curve, auc
)
from scipy.stats import loguniform, uniform

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────

DISEASES = [
    "Monkeypox",
    "Pemphigus",
    "Seborrheic keratosis",
    "Squamous cell carcinoma",
]
IMG_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ─────────────────────────────────────────────────────────────
# SECTION 1 ─ DATA SPLITTING (Stratified, Leak-Free)
# ─────────────────────────────────────────────────────────────

def analyze_dataset_distribution(original_path: str) -> dict:
    """Analyse class counts before splitting."""
    class_counts = {}
    print("Dataset Analysis:")
    print("=" * 50)
    for disease in DISEASES:
        disease_dir = os.path.join(original_path, disease)
        if os.path.exists(disease_dir):
            imgs = [
                f for f in os.listdir(disease_dir)
                if f.lower().endswith(IMG_EXTENSIONS)
            ]
            class_counts[disease] = len(imgs)
            print(f"  {disease}: {len(imgs)} images")
        else:
            class_counts[disease] = 0
            print(f"  {disease}: directory not found")
    print("=" * 50)
    min_s = min(class_counts.values()) if class_counts else 0
    print(f"Minimum samples per class: {min_s}")
    if 0 < min_s < 10:
        print("WARNING: Some classes have very few samples.")
    return class_counts


def split_original_dataset(
    original_path: str,
    train_path: str,
    test_path: str,
    test_size: float = 0.2,
    min_test_samples: int = 2,
) -> bool:
    """
    Stratified train/test split of raw images BEFORE augmentation.
    Copies files into train_path / test_path directory trees.
    Returns True if every class has at least one test sample.
    """
    analyze_dataset_distribution(original_path)

    all_files, all_labels = [], []
    for disease in DISEASES:
        disease_dir = os.path.join(original_path, disease)
        if not os.path.exists(disease_dir):
            continue
        imgs = [
            f for f in os.listdir(disease_dir)
            if f.lower().endswith(IMG_EXTENSIONS)
        ]
        if len(imgs) < 2:
            print(f"Skipping {disease} – only {len(imgs)} image(s).")
            continue
        for img_file in imgs:
            all_files.append((disease, img_file))
            all_labels.append(disease)

    if not all_files:
        raise ValueError("No valid image files found.")

    all_files = np.array(all_files)
    all_labels = np.array(all_labels)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    try:
        train_idx, test_idx = next(sss.split(all_files, all_labels))
    except ValueError as exc:
        print(f"Stratified split failed ({exc}); falling back to random split.")
        idx = np.arange(len(all_files))
        np.random.seed(42)
        np.random.shuffle(idx)
        split_pt = int(len(idx) * (1 - test_size))
        train_idx, test_idx = idx[:split_pt], idx[split_pt:]

    train_files = all_files[train_idx]
    test_files = all_files[test_idx]

    for disease in DISEASES:
        os.makedirs(os.path.join(train_path, disease), exist_ok=True)
        os.makedirs(os.path.join(test_path, disease), exist_ok=True)

    train_counts = {d: 0 for d in DISEASES}
    for disease, img_file in train_files:
        shutil.copy2(
            os.path.join(original_path, disease, img_file),
            os.path.join(train_path, disease, img_file),
        )
        train_counts[disease] += 1

    test_counts = {d: 0 for d in DISEASES}
    for disease, img_file in test_files:
        shutil.copy2(
            os.path.join(original_path, disease, img_file),
            os.path.join(test_path, disease, img_file),
        )
        test_counts[disease] += 1

    print("\nDataset Split Results:")
    print("=" * 50)
    for disease in DISEASES:
        print(f"  {disease}: {train_counts[disease]} train / {test_counts[disease]} test")

    missing = [d for d in DISEASES if test_counts[d] == 0 and train_counts[d] > 0]
    if missing:
        print(f"\nWARNING: No test samples for: {missing}")
        return False

    print("\n✓ All classes have samples in both splits.")
    return True


# ─────────────────────────────────────────────────────────────
# SECTION 2 ─ TRAINING-ONLY AUGMENTATION
# ─────────────────────────────────────────────────────────────

def augment_training_data(
    train_path: str,
    augmented_train_path: str,
    target_per_class: int = 250,
) -> None:
    """
    Offline augmentation of training images only.
    Uses torchvision transforms for consistency with the PyTorch pipeline.
    """
    augment_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
        transforms.ColorJitter(brightness=0.2, contrast=0.1, saturation=0.1),
    ])

    for disease in DISEASES:
        input_dir = os.path.join(train_path, disease)
        output_dir = os.path.join(augmented_train_path, disease)
        os.makedirs(output_dir, exist_ok=True)

        if not os.path.exists(input_dir):
            continue

        img_files = [
            f for f in os.listdir(input_dir)
            if f.lower().endswith(IMG_EXTENSIONS)
        ]
        if not img_files:
            continue

        # Copy originals
        for img_file in img_files:
            shutil.copy2(
                os.path.join(input_dir, img_file),
                os.path.join(output_dir, img_file),
            )

        current_count = len(img_files)
        adjusted_target = min(target_per_class, current_count * 5)

        if current_count >= adjusted_target:
            print(f"  {disease}: {current_count} images (no augmentation needed)")
            continue

        extra_needed = adjusted_target - current_count
        augs_per_image = max(1, extra_needed // current_count)
        print(f"  {disease}: {current_count} → {adjusted_target} (+{extra_needed} augmented)")

        count = 0
        for img_file in img_files:
            if count >= extra_needed:
                break
            img_path = os.path.join(input_dir, img_file)
            try:
                img = Image.open(img_path).convert("RGB")
                img_resized = img.resize((224, 224))
                base_name = Path(img_file).stem
                for j in range(augs_per_image):
                    if count >= extra_needed:
                        break
                    aug_img = augment_transform(img_resized)
                    aug_name = f"{base_name}_aug_{j + 1}.jpg"
                    aug_img.save(os.path.join(output_dir, aug_name), "JPEG", quality=95)
                    count += 1
            except Exception as exc:
                print(f"    Error augmenting {img_file}: {exc}")

        final = len(os.listdir(output_dir))
        print(f"  {disease}: final count = {final}")


# ─────────────────────────────────────────────────────────────
# SECTION 3 ─ PYTORCH DATASET
# ─────────────────────────────────────────────────────────────

class MedicalImageDataset(Dataset):
    """
    Folder-based dataset.  Mirrors torchvision.datasets.ImageFolder
    but tied to the DISEASES list for consistent label ordering.
    """

    def __init__(self, root_dir: str, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples: list[tuple[str, int]] = []
        self.class_to_idx = {d: i for i, d in enumerate(DISEASES)}
        self.idx_to_class = {i: d for d, i in self.class_to_idx.items()}

        for disease in DISEASES:
            disease_dir = os.path.join(root_dir, disease)
            if not os.path.exists(disease_dir):
                continue
            label = self.class_to_idx[disease]
            for fname in os.listdir(disease_dir):
                if fname.lower().endswith(IMG_EXTENSIONS):
                    self.samples.append((os.path.join(disease_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

    def get_class_weights(self) -> torch.Tensor:
        """Inverse-frequency weights for WeightedRandomSampler."""
        labels = np.array([s[1] for s in self.samples])
        counts = np.bincount(labels, minlength=len(DISEASES)).astype(float)
        counts = np.where(counts == 0, 1, counts)          # avoid div-by-zero
        weights = 1.0 / counts
        sample_weights = weights[labels]
        return torch.tensor(sample_weights, dtype=torch.float)


# ─────────────────────────────────────────────────────────────
# SECTION 4 ─ FEATURE EXTRACTION (MobileNetV3)
# ─────────────────────────────────────────────────────────────

class FeatureExtractor:
    """
    MobileNetV3-Large backbone for feature extraction.
    Strips the classifier head; returns 960-d pooled vectors.
    """

    def __init__(self):
        backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        # Remove the classifier; keep features + avgpool
        self.model = nn.Sequential(*list(backbone.children())[:-1])
        self.model.to(DEVICE).eval()

        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ])

    @torch.no_grad()
    def extract_features_batch(
        self, directory: str, batch_size: int = 32
    ) -> tuple[np.ndarray, np.ndarray]:
        dataset = MedicalImageDataset(directory, transform=self.transform)
        if len(dataset) == 0:
            raise ValueError(f"No images found in {directory}")

        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)
        all_features, all_labels = [], []

        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            feats = self.model(imgs)                        # (B, 960, 1, 1)
            feats = feats.view(feats.size(0), -1).cpu().numpy()
            all_features.append(feats)
            all_labels.extend(labels.numpy())

        features_array = np.vstack(all_features)
        labels_array = np.array(all_labels)
        print(f"Extracted features shape: {features_array.shape}")
        return features_array, labels_array

    def extract_single_image(self, image_path: str) -> np.ndarray:
        """Return feature vector for a single image."""
        img = Image.open(image_path).convert("RGB")
        x = self.transform(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            feats = self.model(x).view(1, -1).cpu().numpy()
        return feats


# ─────────────────────────────────────────────────────────────
# SECTION 5 ─ EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────

def check_class_distribution(y_train: np.ndarray, y_test: np.ndarray, label_encoder: LabelEncoder) -> None:
    print("\nClass Distribution:")
    print("-" * 45)
    for i, cls in enumerate(label_encoder.classes_):
        tr = int(np.sum(y_train == i))
        te = int(np.sum(y_test == i))
        warn = "  ⚠ NO TEST SAMPLES" if te == 0 else ""
        print(f"  {cls:<35} train={tr:>4}  test={te:>3}{warn}")
    print()


def plot_confusion_matrix(y_true, y_pred, labels, model_name: str) -> None:
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix – {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()


def calculate_basic_metrics(y_true, y_pred, model_name: str) -> tuple:
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    print(f"\n{model_name} ─ Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}")
    return acc, prec, rec, f1


def print_classification_report(y_true, y_pred, labels, model_name: str) -> None:
    print(f"\n{model_name} Classification Report:")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))


def plot_roc_curve(y_true, y_pred_proba, label_encoder: LabelEncoder, model_name: str) -> None:
    classes = label_encoder.classes_
    plt.figure(figsize=(9, 7))
    for i, cls in enumerate(classes):
        y_bin = (y_true == i).astype(int)
        if y_bin.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin, y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.2f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curves – {model_name}")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_test, y_test, label_encoder, model_name: str, scaler=None, pca=None):
    """Full evaluation pipeline: metrics, report, confusion matrix, ROC."""
    X = X_test.copy()
    if pca is not None:
        X = pca.transform(X)
    if scaler is not None:
        X = scaler.transform(X)

    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)

    acc, prec, rec, f1 = calculate_basic_metrics(y_test, y_pred, model_name)
    print_classification_report(y_test, y_pred, label_encoder.classes_, model_name)
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, model_name)
    plot_roc_curve(y_test, y_proba, label_encoder, model_name)
    return acc


# ─────────────────────────────────────────────────────────────
# SECTION 6 ─ SKLEARN CLASSIFIERS
# ─────────────────────────────────────────────────────────────

# ── KNN (PCA + RobustScaler) ──────────────────────────────────

def train_knn_pca(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[KNN with PCA]")
    pca    = PCA(n_components=0.95)
    scaler = RobustScaler()
    Xtr = scaler.fit_transform(pca.fit_transform(X_train))
    Xte = scaler.transform(pca.transform(X_test))

    model = KNeighborsClassifier(n_neighbors=3, weights="distance",
                                  metric="cosine", algorithm="brute")
    model.fit(Xtr, y_train)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "KNN (PCA)")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "KNN (PCA)")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "KNN (PCA)")
    plot_roc_curve(y_test, y_proba, label_encoder, "KNN (PCA)")
    return model, acc, pca, scaler


# ── KNN (GridSearchCV) ────────────────────────────────────────

def train_knn_grid(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[KNN – GridSearchCV]")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    param_grid = {
        "n_neighbors": [1, 3, 5, 7],
        "weights":     ["uniform", "distance"],
        "metric":      ["euclidean", "cosine"],
    }
    gs = GridSearchCV(KNeighborsClassifier(), param_grid,
                      cv=5, scoring="accuracy", n_jobs=-1, verbose=1)
    gs.fit(Xtr, y_train)
    print(f"Best params: {gs.best_params_}  CV={gs.best_score_:.4f}")

    model   = gs.best_estimator_
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "KNN (GridSearch)")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "KNN (GridSearch)")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "KNN (GridSearch)")
    plot_roc_curve(y_test, y_proba, label_encoder, "KNN (GridSearch)")
    return model, acc, scaler


# ── Logistic Regression ───────────────────────────────────────

def train_logistic_regression(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[Logistic Regression – Manual Tuned]")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    model = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs",
                                max_iter=2000, random_state=42)
    model.fit(Xtr, y_train)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "Logistic Regression")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "Logistic Regression")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "Logistic Regression")
    plot_roc_curve(y_test, y_proba, label_encoder, "Logistic Regression")
    return model, acc, scaler


def train_logistic_regression_randomized(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[Logistic Regression – RandomizedSearchCV]")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    param_dist = {
        "C":        loguniform(0.01, 100),
        "penalty":  ["l2", "elasticnet"],
        "solver":   ["lbfgs", "saga"],
        "max_iter": [1000, 2000],
        "l1_ratio": uniform(0.1, 0.8),
    }
    rs = RandomizedSearchCV(LogisticRegression(random_state=42), param_dist,
                             n_iter=30, cv=5, scoring="accuracy",
                             n_jobs=-1, verbose=1, random_state=42)
    rs.fit(Xtr, y_train)
    print(f"Best params: {rs.best_params_}  CV={rs.best_score_:.4f}")

    model   = rs.best_estimator_
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "LR (RandomSearch)")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "LR (RandomSearch)")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "LR (RandomSearch)")
    plot_roc_curve(y_test, y_proba, label_encoder, "LR (RandomSearch)")
    return model, acc, scaler


# ── SVM ───────────────────────────────────────────────────────

def train_svm(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[SVM – Manual]")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    model = SVC(kernel="rbf", C=10, gamma="scale",
                probability=True, random_state=42)
    model.fit(Xtr, y_train)
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "SVM")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "SVM")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "SVM")
    plot_roc_curve(y_test, y_proba, label_encoder, "SVM")
    return model, acc, scaler


def train_svm_grid(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[SVM – GridSearchCV]")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)

    param_grid = {
        "C":      [0.1, 1, 10, 100],
        "kernel": ["rbf", "linear"],
        "gamma":  ["scale", "auto", 0.01, 0.1],
    }
    gs = GridSearchCV(SVC(probability=True, random_state=42), param_grid,
                      cv=5, scoring="accuracy", n_jobs=-1, verbose=1)
    gs.fit(Xtr, y_train)
    print(f"Best params: {gs.best_params_}  CV={gs.best_score_:.4f}")

    model   = gs.best_estimator_
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "SVM (GridSearch)")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "SVM (GridSearch)")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "SVM (GridSearch)")
    plot_roc_curve(y_test, y_proba, label_encoder, "SVM (GridSearch)")
    return model, acc, scaler


# ── Decision Tree ─────────────────────────────────────────────

def train_decision_tree(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[Decision Tree – GridSearchCV + PCA]")
    pca    = PCA(n_components=0.95)
    scaler = RobustScaler()
    Xtr = scaler.fit_transform(pca.fit_transform(X_train))
    Xte = scaler.transform(pca.transform(X_test))

    param_grid = {
        "max_depth":        [3, 5, 10, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "criterion":        ["gini", "entropy"],
    }
    gs = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid,
                      cv=5, scoring="accuracy", n_jobs=-1, verbose=1)
    gs.fit(Xtr, y_train)
    print(f"Best params: {gs.best_params_}  CV={gs.best_score_:.4f}")

    model   = gs.best_estimator_
    y_pred  = model.predict(Xte)
    y_proba = model.predict_proba(Xte)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "Decision Tree")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "Decision Tree")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "Decision Tree")
    plot_roc_curve(y_test, y_proba, label_encoder, "Decision Tree")
    return model, acc, pca, scaler


# ── Random Forest ─────────────────────────────────────────────

def train_random_forest(X_train, y_train, X_test, y_test, label_encoder):
    print("\n[Random Forest – GridSearchCV]")
    param_grid = {
        "n_estimators": [100, 200, 300],
        "max_depth":    [10, 20, None],
        "max_features": ["sqrt", "log2"],
    }
    gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid,
                      cv=5, scoring="accuracy", n_jobs=-1, verbose=1)
    gs.fit(X_train, y_train)
    print(f"Best params: {gs.best_params_}  CV={gs.best_score_:.4f}")

    model   = gs.best_estimator_
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc, *_ = calculate_basic_metrics(y_test, y_pred, "Random Forest")
    print_classification_report(y_test, y_pred, label_encoder.classes_, "Random Forest")
    plot_confusion_matrix(y_test, y_pred, label_encoder.classes_, "Random Forest")
    plot_roc_curve(y_test, y_proba, label_encoder, "Random Forest")
    return model, acc


# ─────────────────────────────────────────────────────────────
# SECTION 7 ─ PYTORCH CNN (Transfer Learning)
# ─────────────────────────────────────────────────────────────

class MedicalCNN(nn.Module):
    """MobileNetV3-Large fine-tuned classifier."""

    def __init__(self, num_classes: int = 4, freeze_backbone: bool = True):
        super().__init__()
        backbone = models.mobilenet_v3_large(
            weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2
        )
        if freeze_backbone:
            for p in backbone.parameters():
                p.requires_grad = False
        # Replace final classifier
        in_features = backbone.classifier[3].in_features
        backbone.classifier[3] = nn.Linear(in_features, num_classes)
        self.net = backbone

    def forward(self, x):
        return self.net(x)

    def unfreeze_backbone(self, n_layers: int = 3):
        """Unfreeze the last n feature-extraction blocks for fine-tuning."""
        layers = list(self.net.features.children())
        for layer in layers[-n_layers:]:
            for p in layer.parameters():
                p.requires_grad = True


class CNNTrainer:
    def __init__(
        self,
        model: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        test_loader: DataLoader,
        num_classes: int = 4,
        lr: float = 1e-4,
    ):
        self.model       = model.to(DEVICE)
        self.train_loader = train_loader
        self.val_loader   = val_loader
        self.test_loader  = test_loader
        self.num_classes  = num_classes

        # Class-weighted cross-entropy
        counts  = np.bincount(
            [s[1] for s in train_loader.dataset.samples],
            minlength=num_classes,
        ).astype(float)
        weights = torch.tensor(1.0 / np.where(counts == 0, 1, counts),
                               dtype=torch.float).to(DEVICE)
        self.criterion = nn.CrossEntropyLoss(weight=weights)
        self.optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=lr
        )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="max", factor=0.5, patience=5, verbose=True
        )
        self.history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}

    # ── single epoch ─────────────────────────────────────────
    def _run_epoch(self, loader: DataLoader, train: bool):
        self.model.train(train)
        total_loss, correct, total = 0.0, 0, 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for imgs, labels in loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                if train:
                    self.optimizer.zero_grad()
                out  = self.model(imgs)
                loss = self.criterion(out, labels)
                if train:
                    loss.backward()
                    self.optimizer.step()
                total_loss += loss.item() * imgs.size(0)
                correct    += (out.argmax(1) == labels).sum().item()
                total      += imgs.size(0)
        return total_loss / total, correct / total

    # ── full training loop ────────────────────────────────────
    def train(self, epochs: int = 50, patience: int = 10, save_path: str = "best_cnn.pt"):
        best_val_acc, patience_counter = 0.0, 0

        for epoch in range(1, epochs + 1):
            tr_loss, tr_acc = self._run_epoch(self.train_loader, train=True)
            va_loss, va_acc = self._run_epoch(self.val_loader,   train=False)
            self.scheduler.step(va_acc)

            self.history["train_acc"].append(tr_acc)
            self.history["val_acc"].append(va_acc)
            self.history["train_loss"].append(tr_loss)
            self.history["val_loss"].append(va_loss)

            print(
                f"Epoch {epoch:>3}/{epochs}  "
                f"loss={tr_loss:.4f}  acc={tr_acc:.4f}  "
                f"val_loss={va_loss:.4f}  val_acc={va_acc:.4f}"
            )

            if va_acc > best_val_acc:
                best_val_acc = va_acc
                patience_counter = 0
                torch.save(self.model.state_dict(), save_path)
                print(f"  ✓ Saved best model (val_acc={best_val_acc:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"Early stopping at epoch {epoch}.")
                    break

        # Reload best weights
        self.model.load_state_dict(torch.load(save_path, map_location=DEVICE))
        return self.history

    # ── evaluation ────────────────────────────────────────────
    def evaluate(self, label_encoder: LabelEncoder):
        self.model.eval()
        all_preds, all_labels, all_probs = [], [], []
        with torch.no_grad():
            for imgs, labels in self.test_loader:
                imgs = imgs.to(DEVICE)
                out  = self.model(imgs)
                probs = torch.softmax(out, dim=1).cpu().numpy()
                preds = out.argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())
                all_probs.extend(probs)

        y_pred  = np.array(all_preds)
        y_true  = np.array(all_labels)
        y_proba = np.array(all_probs)

        acc, *_ = calculate_basic_metrics(y_true, y_pred, "CNN (MobileNetV3)")
        print_classification_report(y_true, y_pred, label_encoder.classes_, "CNN (MobileNetV3)")
        plot_confusion_matrix(y_true, y_pred, label_encoder.classes_, "CNN (MobileNetV3)")
        plot_roc_curve(y_true, y_proba, label_encoder, "CNN (MobileNetV3)")
        return acc

    # ── training curves ───────────────────────────────────────
    def plot_history(self):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].plot(self.history["train_acc"], label="Train")
        axes[0].plot(self.history["val_acc"],   label="Val")
        axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].grid(True)

        axes[1].plot(self.history["train_loss"], label="Train")
        axes[1].plot(self.history["val_loss"],   label="Val")
        axes[1].set_title("Loss"); axes[1].legend(); axes[1].grid(True)
        plt.tight_layout(); plt.show()


def build_cnn_dataloaders(
    train_dir: str,
    test_dir: str,
    batch_size: int = 16,
    val_split: float = 0.2,
):
    """Build train / val / test DataLoaders with proper augmentation."""
    train_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])

    full_train = MedicalImageDataset(train_dir, transform=train_tf)
    n_val   = int(len(full_train) * val_split)
    n_train = len(full_train) - n_val
    train_ds, val_ds = torch.utils.data.random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    # Assign eval transform to val split
    val_ds.dataset.transform = eval_tf

    test_ds = MedicalImageDataset(test_dir, transform=eval_tf)

    # Balanced sampler for training
    train_weights = full_train.get_class_weights()[
        [full_train.samples[i][1] for i in train_ds.indices]
    ]
    sampler = WeightedRandomSampler(train_weights, len(train_ds), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,   num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,   num_workers=0)

    print(f"Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader


# ─────────────────────────────────────────────────────────────
# SECTION 8 ─ INFERENCE HELPER
# ─────────────────────────────────────────────────────────────

def predict_single_image(
    image_path: str,
    model,                        # sklearn or PyTorch nn.Module
    label_encoder: LabelEncoder,
    feature_extractor: FeatureExtractor | None = None,
    scaler=None,
    pca=None,
    is_pytorch_model: bool = False,
):
    """
    Unified inference for both sklearn classifiers and PyTorch CNN.
    """
    if is_pytorch_model:
        tf_eval = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])
        img = Image.open(image_path).convert("RGB")
        x   = tf_eval(img).unsqueeze(0).to(DEVICE)
        model.eval()
        with torch.no_grad():
            out   = model(x)
            probs = torch.softmax(out, dim=1).cpu().numpy()[0]
        pred_idx = int(probs.argmax())
        predicted_class = label_encoder.classes_[pred_idx]
    else:
        if feature_extractor is None:
            raise ValueError("feature_extractor required for sklearn models.")
        feats = feature_extractor.extract_single_image(image_path)
        if pca is not None:
            feats = pca.transform(feats)
        if scaler is not None:
            feats = scaler.transform(feats)
        pred_idx = model.predict(feats)[0]
        probs    = model.predict_proba(feats)[0]
        predicted_class = label_encoder.classes_[pred_idx]

    confidence = {label_encoder.classes_[i]: float(probs[i]) for i in range(len(probs))}
    print(f"\n✅  Image : {image_path}")
    print(f"   Prediction : {predicted_class}")
    for cls, p in sorted(confidence.items(), key=lambda x: -x[1]):
        print(f"   {cls:<35} {p:.4f}")
    return predicted_class, confidence


# ─────────────────────────────────────────────────────────────
# SECTION 9 ─ MODEL COMPARISON PLOT
# ─────────────────────────────────────────────────────────────

def plot_model_comparison(accuracies: dict) -> None:
    df = (pd.DataFrame(list(accuracies.items()), columns=["Model", "Accuracy"])
          .sort_values("Accuracy", ascending=False))
    print("\n" + "=" * 45)
    print("MODEL ACCURACY COMPARISON")
    print("=" * 45)
    print(df.to_string(index=False))

    plt.figure(figsize=(11, 5))
    bars = plt.bar(df["Model"], df["Accuracy"], color=plt.cm.viridis(np.linspace(0.2, 0.8, len(df))))
    plt.ylim(0, 1.05)
    for bar, val in zip(bars, df["Accuracy"]):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.title("Model Accuracy Comparison")
    plt.ylabel("Accuracy")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────

def main():
    # ── Paths (adjust for your environment) ──────────────────
    original_path        = "/kaggle/input/four-diseases-dataset/Dataset"
    train_path           = "/kaggle/working/train_dataset"
    test_path            = "/kaggle/working/test_dataset"
    augmented_train_path = "/kaggle/working/augmented_train_dataset"

    # ── Step 1 : Split ───────────────────────────────────────
    print("=" * 60)
    print("STEP 1 – Data Splitting")
    print("=" * 60)
    ok = split_original_dataset(original_path, train_path, test_path, test_size=0.15)
    if not ok:
        print("Retrying with test_size=0.10 …")
        ok = split_original_dataset(original_path, train_path, test_path, test_size=0.10)
        if not ok:
            print("Could not create valid split. Exiting.")
            return

    # ── Step 2 : Augment training only ───────────────────────
    print("\n" + "=" * 60)
    print("STEP 2 – Training-Only Augmentation")
    print("=" * 60)
    augment_training_data(train_path, augmented_train_path, target_per_class=250)

    # ── Step 3 : Feature extraction ──────────────────────────
    print("\n" + "=" * 60)
    print("STEP 3 – Feature Extraction (MobileNetV3)")
    print("=" * 60)
    extractor = FeatureExtractor()
    X_train, y_train_raw = extractor.extract_features_batch(augmented_train_path)
    X_test,  y_test_raw  = extractor.extract_features_batch(test_path)

    # ── Step 4 : Label encoding ──────────────────────────────
    print("\n" + "=" * 60)
    print("STEP 4 – Label Encoding")
    print("=" * 60)
    le = LabelEncoder()
    le.classes_ = np.array(DISEASES)       # deterministic ordering
    y_train = le.transform([DISEASES[i] for i in y_train_raw]) if y_train_raw.dtype != int \
              else y_train_raw
    # The extractor returns integer labels directly via MedicalImageDataset
    y_train = y_train_raw
    y_test  = y_test_raw
    check_class_distribution(y_train, y_test, le)

    # ── Step 5 : Sklearn classifiers ─────────────────────────
    print("\n" + "=" * 60)
    print("STEP 5 – Training Sklearn Classifiers")
    print("=" * 60)
    accuracies = {}

    knn_model,   knn_acc,   knn_pca,    knn_scaler = train_knn_pca(X_train, y_train, X_test, y_test, le)
    accuracies["KNN (PCA)"] = knn_acc

    lr_model,    lr_acc,    lr_scaler               = train_logistic_regression(X_train, y_train, X_test, y_test, le)
    accuracies["Logistic Regression"] = lr_acc

    svm_model,   svm_acc,   svm_scaler              = train_svm(X_train, y_train, X_test, y_test, le)
    accuracies["SVM"] = svm_acc

    dt_model,    dt_acc,    dt_pca,     dt_scaler   = train_decision_tree(X_train, y_train, X_test, y_test, le)
    accuracies["Decision Tree"] = dt_acc

    rf_model,    rf_acc                             = train_random_forest(X_train, y_train, X_test, y_test, le)
    accuracies["Random Forest"] = rf_acc

    # ── Step 6 : CNN (transfer learning) ─────────────────────
    print("\n" + "=" * 60)
    print("STEP 6 – CNN Fine-Tuning (MobileNetV3)")
    print("=" * 60)
    train_loader, val_loader, test_loader = build_cnn_dataloaders(
        augmented_train_path, test_path, batch_size=16
    )
    cnn_model = MedicalCNN(num_classes=len(DISEASES), freeze_backbone=True)
    trainer   = CNNTrainer(cnn_model, train_loader, val_loader, test_loader,
                           num_classes=len(DISEASES), lr=1e-4)
    trainer.train(epochs=50, patience=10, save_path="best_cnn.pt")
    trainer.plot_history()
    cnn_acc = trainer.evaluate(le)
    accuracies["CNN (MobileNetV3)"] = cnn_acc

    # ── Step 7 : Optional fine-tune (unfreeze last 3 blocks) ─
    print("\nFine-tuning last 3 backbone blocks …")
    cnn_model.unfreeze_backbone(n_layers=3)
    trainer.optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, cnn_model.parameters()), lr=1e-5
    )
    trainer.train(epochs=20, patience=7, save_path="best_cnn_finetuned.pt")
    trainer.plot_history()
    cnn_ft_acc = trainer.evaluate(le)
    accuracies["CNN Fine-Tuned"] = cnn_ft_acc

    # ── Step 8 : Comparison ───────────────────────────────────
    plot_model_comparison(accuracies)

    # ── Step 9 : Save artefacts ───────────────────────────────
    best_name = max(accuracies, key=accuracies.get)
    print(f"\n🏆  Best model: {best_name}  ({accuracies[best_name]:.4f})")

    with open("/kaggle/working/label_encoder.pkl", "wb") as f:
        pickle.dump(le, f)
    with open("/kaggle/working/best_sklearn_model.pkl", "wb") as f:
        pickle.dump(
            {"name": best_name, "model": lr_model, "scaler": lr_scaler},
            f,
        )
    print("Artefacts saved to /kaggle/working/")


if __name__ == "__main__":
    main()